In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('WELFake_Dataset.csv')

In [3]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
df.shape

(72134, 4)

In [5]:
df.columns

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='str')

In [6]:
df.dtypes

Unnamed: 0    int64
title           str
text            str
label         int64
dtype: object

In [7]:
df['Fake News']=df['title']+' '+df['text']

In [8]:
df=df.drop(columns=['Unnamed: 0','title','text'])

In [9]:
df.head()

,label,Fake News
0,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,NaN
2,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,1,SATAN 2: Russia unvelis an image of its terrif...


In [10]:
df.isnull().sum()

label          0
Fake News    597
dtype: int64

In [11]:
df=df.dropna()

In [12]:
df.isnull().sum()

label        0
Fake News    0
dtype: int64

In [13]:
df.shape

(71537, 2)

In [14]:
df.duplicated().sum()

np.int64(8416)

In [15]:
df[df.duplicated(keep=False)]

,label,Fake News
4,1,SATAN 2: Russia unvelis an image of its terrif...
5,1,About Time! Christian Group Sues Amazon and SP...
7,1,HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...
8,1,Sports Bar Owner Bans NFL Games…Will Show Only...
13,1,WATCH: HILARIOUS AD Calls Into Question Health...
...,...,...
72116,1,SAY “HELLO” TO YOUR NEW NEIGHBORS! Clooney Beg...
72118,1,LEFTY MEDIA DESPERATELY Tries To Bury Trump Bu...
72125,1,WOW! JILL STEIN’S ‘FIRESIDE CHAT’ Exposes Her ...
72128,1,JUDGE JEANINE SOUNDS FREE SPEECH ALARM: “They ...


In [16]:
df=df.drop_duplicates()

In [17]:
df.shape

(63121, 2)

In [18]:
## Check whether the dataset is an imbalance dataset or not
df['label'].value_counts()

label
0    34791
1    28330
Name: count, dtype: int64

In [19]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\panta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [21]:
stop_words = set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

In [ ]:
def preprocess(text):
    ## Lower the text
    text = text.lower()
    ## remove html tags
    text = re.sub(r'<.*?>', '', text)
    ## remove urls
    text = re.sub(r'http\S+|www\S+', '', text)
    ## Remove special characters and numbers
    text = re.sub(r'[^a-z\s]+', '', text)
    ## Remove extra spaces 
    text = re.sub(r'\s+', ' ', text).strip()
    ## Remove stopwords and apply lemmatization
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text

In [23]:
df['Fake News'] = df['Fake News'].apply(preprocess)

In [24]:
df.head()

,label,Fake News
0,1,law enforcement high alert following threat co...
2,1,unbelievable obamas attorney general say charl...
3,0,bobby jindal raised hindu us story christian c...
4,1,satan russia unvelis image terrifying new supe...
5,1,time christian group sue amazon splc designati...


In [25]:
## Independent column
X=df['Fake News']
## Dependent Column
y=df['label']

In [26]:
## Apply Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [27]:
from nltk.tokenize import word_tokenize

In [28]:
from gensim.models import Word2Vec

## Train Data tokenization
X_train_tokens=X_train.apply(lambda x: word_tokenize(str(x)))

## Train word2vec model
word2vec_model=Word2Vec(sentences=X_train_tokens,vector_size=100,window=5,min_count=2,workers=4)

In [29]:
word2vec_model.wv['news']

array([-0.0278815 , -0.3848802 , -0.01173616, -3.3403194 , -2.0160182 ,
        0.44832546,  5.55242   , -0.8809219 , -0.539604  , -1.854226  ,
        0.4666654 , -1.8449923 , -3.2583742 , -2.6344936 ,  0.44072694,
        2.536185  ,  0.51351285,  3.079081  , -0.95869493, -0.819787  ,
        0.567344  ,  1.098278  ,  1.8051003 , -2.0472722 ,  2.7885823 ,
        3.2784832 , -1.3172157 ,  2.4531603 ,  1.8277881 ,  1.4008379 ,
       -2.800357  ,  1.3337648 ,  1.0925491 , -2.3368523 , -3.1430287 ,
       -0.35891914, -2.3919878 ,  0.36700726, -0.70117843,  1.873327  ,
        1.9686537 ,  1.6392004 ,  0.10873155,  4.943434  ,  1.4925677 ,
       -1.1029315 ,  2.7619286 , -1.9544363 ,  0.6634723 , -0.30787706,
       -1.7596034 ,  1.6203034 ,  1.5797095 ,  0.1903458 , -1.2537506 ,
       -2.982244  , -2.5469892 ,  2.780757  ,  3.2374446 , -0.40348986,
        0.85456   , -2.8527014 , -1.5447522 , -1.96545   ,  0.09595846,
       -0.8116352 , -1.1847044 , -1.9196231 , -2.5212705 ,  1.88

In [30]:
word2vec_model.wv.most_similar('news')

[('newsthe', 0.7362796068191528),
 ('newsit', 0.6727791428565979),
 ('newstrump', 0.6623924970626831),
 ('newswatch', 0.6487988233566284),
 ('newsand', 0.6346392631530762),
 ('newsfacebook', 0.6223862171173096),
 ('newsaccording', 0.6120315790176392),
 ('newsi', 0.6038673520088196),
 ('newshere', 0.5868428349494934),
 ('newsthis', 0.5777928233146667)]

In [31]:
X_test_tokens=X_test.apply(lambda x: word_tokenize(str(x)))

In [32]:
def get_sentence_vector(sentence):
    vectors=[]

    for word in sentence:
        if word in word2vec_model.wv:
            vectors.append(word2vec_model.wv[word])

    if(len(vectors))==0:
        return np.zeros(100)
    
    return np.mean(vectors,axis=0)

In [33]:
X_train_vectors=np.array([get_sentence_vector(sentence) for sentence in X_train_tokens])
X_test_vectors=np.array([get_sentence_vector(sentence) for sentence in X_test_tokens])

In [34]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression().fit(X_train_vectors,y_train)

In [35]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred=lr_model.predict(X_test_vectors)

In [36]:
print("Accuracy Score: ", accuracy_score(y_test,y_pred))

Accuracy Score:  0.8974257425742574


In [37]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.91      0.90      0.91      6947
           1       0.88      0.89      0.89      5678

    accuracy                           0.90     12625
   macro avg       0.90      0.90      0.90     12625
weighted avg       0.90      0.90      0.90     12625

